# Desafio 2 — SCD Tipo 2: Histórico de Taxas de Câmbio (ExchangeRate-API)

**Conceitos praticados:** Slowly Changing Dimensions Tipo 2, `MERGE INTO` com múltiplos `WHEN MATCHED`,
lógica condicional de update + insert, Delta Lake como base de dados histórico versionado.

**Seções do exame cobertas:** S2 (Data Ingestion) e S3 (Data Transformation and Modeling)

**API:** https://open.er-api.com/v6/latest/USD — sem cadastro, atualização diária, 160+ moedas.

### O que é SCD Tipo 2?
Imagine o RH de uma empresa guardando o histórico de salários. Quando o salário muda, eles não
apagam o registro antigo — eles **fecham** o período passado (`valid_to = ontem`) e **abrem**
um novo (`valid_to = NULL`). Assim você sempre consegue responder: "qual era o salário no dia X?"

Aqui fazemos o mesmo com taxas de câmbio: cada vez que a taxa muda, o registro antigo é preservado
e um novo é criado. A coluna `is_current = true` indica o registro vigente.

### 3 cenários do MERGE:
1. **Par novo:** INSERT com `valid_from = hoje`, `valid_to = NULL`, `is_current = true`
2. **Par existente, taxa igual:** nenhuma ação (skip)
3. **Par existente, taxa mudou:** UPDATE no antigo (`valid_to = ontem`, `is_current = false`) + INSERT novo

> ⚠️ Se o outbound internet estiver bloqueado na Free Edition, use o **Plano B** documentado na Célula 6.

In [ ]:
%sql
-- Passo 0: criar o schema exchange (execute uma única vez)
CREATE SCHEMA IF NOT EXISTS workshop_weather.exchange
  COMMENT 'Taxas de câmbio — Frankfurter e ExchangeRate-API';

In [ ]:
# ── Imports e configuração ──────────────────────────────────────────────────
import requests
import json
import uuid
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    DateType, TimestampType, BooleanType, LongType
)

CATALOG = "workshop_weather"
SCHEMA  = "exchange"
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.raw_exchangerate"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.rates_scd2"
TABLE_GOLD   = f"{CATALOG}.{SCHEMA}.current_rates_snapshot"

# Moedas de interesse — filtramos para não gravar as 160+ moedas do payload
CURRENCIES_OF_INTEREST = {"BRL", "EUR", "GBP", "JPY", "ARS", "CLP", "COP", "MXN"}

print(f"Bronze: {TABLE_BRONZE}")
print(f"Silver: {TABLE_SILVER}")
print(f"Gold:   {TABLE_GOLD}")
print(f"Moedas monitoradas: {sorted(CURRENCIES_OF_INTEREST)}")

In [ ]:
# ── Função de extração da API ────────────────────────────────────────────────
def fetch_exchangerate(base: str = "USD") -> dict | None:
    """
    Busca as taxas atuais da ExchangeRate-API Open Access.
    Retorna o dicionário JSON completo ou None em caso de falha.
    """
    url = f"https://open.er-api.com/v6/latest/{base}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get("result") != "success":
            print(f"[AVISO] API retornou result != success: {data.get('result')}")
            return None
        return data
    except Exception as e:
        print(f"[ERRO] Falha ao acessar {url}: {e}")
        return None

# Teste de conectividade
test = fetch_exchangerate("USD")
if test:
    print(f"[OK] API acessível. Última atualização: {test.get('time_last_update_utc')}")
    print(f"     Total de moedas retornadas: {len(test.get('rates', {}))}")
else:
    print("[AVISO] API inacessível. Siga para o Plano B (Célula 6).")

### ⚠️ Plano B — API inacessível (outbound bloqueado)

Execute o script abaixo **localmente** na sua máquina para gerar 2 snapshots simulando uma mudança de taxa.
Depois faça upload de ambos os arquivos para o Volume e ajuste `USE_PLAN_B = True` na Célula 7.

```python
# Execute LOCALMENTE (não no Databricks):
import requests, json, copy

r = requests.get("https://open.er-api.com/v6/latest/USD")
snapshot_1 = r.json()

# Simula variação de taxa no "dia seguinte" — modifica BRL e ARS
snapshot_2 = copy.deepcopy(snapshot_1)
snapshot_2["rates"]["BRL"] = round(snapshot_1["rates"]["BRL"] + 0.15, 4)
snapshot_2["rates"]["ARS"] = round(snapshot_1["rates"]["ARS"] + 20.0, 4)

with open("exchangerate_day1.json", "w") as f:
    json.dump(snapshot_1, f)
with open("exchangerate_day2.json", "w") as f:
    json.dump(snapshot_2, f)

print("Arquivos gerados: exchangerate_day1.json e exchangerate_day2.json")
```

Faça upload para: **Data > Volumes > workshop_weather > bronze > landing > exchangerate**

Para simular os 2 dias:
1. Defina `USE_PLAN_B = True` e `PLAN_B_FILE = ".../exchangerate_day1.json"` → execute o notebook
2. Troque `PLAN_B_FILE` para `exchangerate_day2.json` → execute novamente
3. Observe o SCD Tipo 2 em ação!

In [ ]:
# ── Configuração Plano B ─────────────────────────────────────────────────────
USE_PLAN_B  = False
PLAN_B_FILE = "/Volumes/workshop_weather/bronze/landing/exchangerate/exchangerate_day1.json"
# Para simular o dia 2, troque para: exchangerate_day2.json
# ────────────────────────────────────────────────────────────────────────────

batch_id     = str(uuid.uuid4())
ingestion_ts = datetime.utcnow()

# Obtém o payload (API ou Plano B)
if not USE_PLAN_B:
    raw_data = fetch_exchangerate("USD")
    if raw_data is None:
        raise RuntimeError("API falhou. Ative USE_PLAN_B = True e siga as instruções.")
    source_ref = f"https://open.er-api.com/v6/latest/USD"
else:
    print(f"[PLANO B] Lendo: {PLAN_B_FILE}")
    raw_text = spark.read.format("text").load(PLAN_B_FILE).collect()[0][0]
    raw_data  = json.loads(raw_text)
    source_ref = PLAN_B_FILE

# Filtra apenas as moedas de interesse antes de gravar
# (evita persistir 160+ moedas desnecessárias na Bronze)
filtered_rates = {k: v for k, v in raw_data["rates"].items()
                  if k in CURRENCIES_OF_INTEREST}

# Grava 1 linha por moeda na Bronze
rows_bronze = [
    (
        raw_data["base_code"],          # base_currency
        currency,                        # target_currency
        float(rate),                     # exchange_rate
        raw_data["time_last_update_utc"],# reference_time (string)
        json.dumps({                     # raw_payload
            "base_code": raw_data["base_code"],
            "time_last_update_utc": raw_data["time_last_update_utc"],
            "rates": {currency: rate}
        }),
        batch_id,
        ingestion_ts,
        source_ref
    )
    for currency, rate in filtered_rates.items()
]

bronze_schema = StructType([
    StructField("base_currency",         StringType(),    False),
    StructField("target_currency",       StringType(),    False),
    StructField("exchange_rate",         DoubleType(),    True),
    StructField("reference_time_utc",    StringType(),    True),
    StructField("raw_payload",           StringType(),    False),
    StructField("_batch_id",             StringType(),    False),
    StructField("_ingestion_timestamp",  TimestampType(), False),
    StructField("_source_url",           StringType(),    True),
])

df_bronze = spark.createDataFrame(rows_bronze, schema=bronze_schema)
df_bronze.write.format("delta").mode("append").saveAsTable(TABLE_BRONZE)

print(f"[OK] Bronze gravada. +{len(rows_bronze)} linhas (batch_id={batch_id[:8]}...)")
spark.sql(f"SELECT COUNT(*) as total FROM {TABLE_BRONZE}").show()

In [ ]:
# ── Cria a tabela Silver SCD Tipo 2 se não existir ──────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_SILVER} (
    scd_id               BIGINT    GENERATED ALWAYS AS IDENTITY,
    base_currency        STRING    NOT NULL,
    target_currency      STRING    NOT NULL,
    exchange_rate        DOUBLE,
    valid_from           DATE      NOT NULL,
    valid_to             DATE,
    is_current           BOOLEAN   NOT NULL,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Taxas de câmbio — SCD Tipo 2: histórico completo de cada par de moedas'
""")

print(f"[OK] Tabela Silver SCD2 pronta: {TABLE_SILVER}")
spark.sql(f"DESCRIBE {TABLE_SILVER}").show(truncate=False)

## Passo 2 — Silver SCD Tipo 2: lógica do MERGE

O MERGE abaixo implementa os **3 cenários do SCD Tipo 2**:

| Cenário | Condição | Ação |
|---|---|---|
| Par novo | `NOT MATCHED` | INSERT com `valid_from = hoje`, `valid_to = NULL`, `is_current = true` |
| Taxa igual | `MATCHED AND taxa = taxa` | Nenhuma ação (skip) |
| Taxa mudou | `MATCHED AND taxa ≠ taxa` | **2 operações:** (1) UPDATE antigo → fecha período; (2) INSERT novo registro |

> **Por que `valid_to = NULL` e não uma data futura (ex: 9999-12-31)?**
> Porque `NULL` é semanticamente correto: significa "sem data de fim definida".
> Uma data futura arbitrária exigiria limpeza quando o sistema passar dessa data.
> Com `NULL`, a query `WHERE valid_to IS NULL` sempre retorna o registro vigente corretamente.

In [ ]:
from pyspark.sql.functions import current_date, date_sub, col, lit

# ── Prepara o DataFrame de staging com os dados mais recentes da Bronze ──────
# Pega apenas o batch mais recente por par de moedas
from pyspark.sql.window import Window

window_latest = Window.partitionBy("base_currency", "target_currency") \
                      .orderBy(F.col("_ingestion_timestamp").desc())

df_latest = (
    spark.read.format("delta").table(TABLE_BRONZE)
    .withColumn("rn", F.row_number().over(window_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("snapshot_date", F.to_date(F.col("_ingestion_timestamp")))
    .select("base_currency", "target_currency", "exchange_rate",
            "snapshot_date", "_ingestion_timestamp")
)

df_latest.createOrReplaceTempView("scd2_staging")

print(f"[INFO] Staging com {df_latest.count()} pares de moedas (batch mais recente):")
df_latest.show()

In [ ]:
# ── SCD Tipo 2 — Passo 1: fechar registros ANTIGOS cujas taxas mudaram ───────
#
# Não conseguimos fazer "UPDATE + INSERT" em um único MERGE no Delta Lake
# para o SCD Tipo 2 completo. A estratégia é:
#   Passo 1: UPDATE nos registros is_current=true cuja taxa mudou → fecha o período
#   Passo 2: INSERT dos novos registros para os pares que mudaram (ou são novos)

spark.sql(f"""
MERGE INTO {TABLE_SILVER} AS target
USING scd2_staging AS source
ON  target.base_currency   = source.base_currency
AND target.target_currency = source.target_currency
AND target.is_current      = true

-- Cenário: taxa mudou → fecha o registro antigo
WHEN MATCHED AND target.exchange_rate != source.exchange_rate THEN
    UPDATE SET
        valid_to  = date_sub(source.snapshot_date, 1),
        is_current = false
""")

print("[OK] Passo 1 concluído: registros antigos fechados (onde taxa mudou).")

In [ ]:
# ── SCD Tipo 2 — Passo 2: INSERT de registros novos (par novo ou taxa mudou) ─
#
# Insere quando:
#   (a) O par não existe ainda na Silver (NOT MATCHED)
#   (b) O par existe mas a taxa mudou → is_current=false após o Passo 1,
#       então NOT MATCHED ON is_current=true

spark.sql(f"""
MERGE INTO {TABLE_SILVER} AS target
USING scd2_staging AS source
ON  target.base_currency   = source.base_currency
AND target.target_currency = source.target_currency
AND target.is_current      = true

-- Cenário: par novo OU taxa mudou (registro antigo já foi fechado no Passo 1)
WHEN NOT MATCHED THEN
    INSERT (base_currency, target_currency, exchange_rate,
            valid_from, valid_to, is_current, _ingestion_timestamp)
    VALUES (source.base_currency, source.target_currency, source.exchange_rate,
            source.snapshot_date, NULL, true, source._ingestion_timestamp)
""")

print("[OK] Passo 2 concluído: novos registros inseridos.")

# Validação
print("\n[INFO] Estado atual da Silver SCD2:")
spark.sql(f"""
SELECT
    base_currency,
    target_currency,
    exchange_rate,
    valid_from,
    valid_to,
    is_current
FROM {TABLE_SILVER}
ORDER BY base_currency, target_currency, valid_from DESC
""").show(30, truncate=False)

In [ ]:
# ── Validação do SCD Tipo 2 ──────────────────────────────────────────────────
print("=" * 60)
print("VALIDAÇÃO SCD TIPO 2")
print("=" * 60)

# Total de registros vigentes (deve ser igual ao número de pares monitorados)
vigentes = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_SILVER} WHERE is_current = true").collect()[0][0]
historico = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_SILVER} WHERE is_current = false").collect()[0][0]
print(f"Registros vigentes  (is_current=true):  {vigentes}  (esperado: {len(CURRENCIES_OF_INTEREST)})")
print(f"Registros históricos (is_current=false): {historico}")

# Pares com mais de 1 versão (evidência de mudança de taxa)
print("\nPares com histórico de mudança (is_current=true e false coexistem):")
spark.sql(f"""
SELECT base_currency, target_currency, COUNT(*) as versoes
FROM {TABLE_SILVER}
GROUP BY base_currency, target_currency
HAVING COUNT(*) > 1
ORDER BY versoes DESC
""").show()

# Linha do tempo de um par específico
print("Linha do tempo completa do par USD → BRL:")
spark.sql(f"""
SELECT exchange_rate, valid_from, valid_to, is_current
FROM {TABLE_SILVER}
WHERE base_currency = 'USD' AND target_currency = 'BRL'
ORDER BY valid_from DESC
""").show(truncate=False)

In [ ]:
# ── Gold: snapshot atual (apenas registros vigentes) ────────────────────────
spark.sql(f"""
CREATE OR REPLACE TABLE {TABLE_GOLD}
COMMENT 'Snapshot atual de taxas de câmbio — apenas is_current = true'
AS
SELECT
    base_currency,
    target_currency,
    exchange_rate,
    valid_from   AS rate_since,
    _ingestion_timestamp
FROM {TABLE_SILVER}
WHERE is_current = true
ORDER BY base_currency, target_currency
""")

print(f"[OK] Gold gravada: {TABLE_GOLD}")
display(spark.table(TABLE_GOLD))

In [ ]:
print("""
PERGUNTAS DE FIXAÇÃO (responda mentalmente):

1. Por que o SCD Tipo 2 usa 2 operações separadas (UPDATE + INSERT)
   em vez de um único MERGE?

2. Se você executar este notebook 3x seguidas SEM mudar nenhuma taxa,
   quantos registros a Silver deve ter? O número cresce? Por quê?

3. Qual é a diferença entre SCD Tipo 1 e SCD Tipo 2?
   Em qual cenário de negócio o Tipo 1 seria suficiente?

4. Por que valid_to = NULL é preferível a valid_to = '9999-12-31'?

5. Como você usaria a Silver SCD2 para responder:
   "Qual era a taxa USD/BRL no dia 2024-03-15?"
   Escreva a query SQL mentalmente antes de checar a resposta.

   RESPOSTA:
   SELECT exchange_rate FROM rates_scd2
   WHERE base_currency = 'USD' AND target_currency = 'BRL'
     AND valid_from <= DATE '2024-03-15'
     AND (valid_to IS NULL OR valid_to >= DATE '2024-03-15');
""")